# Notebook 04 — Citation Graph & GNN Scorer

**Purpose**: Construct a citation graph from Semantic Scholar S2AG API, train a Graph Neural Network (GNN) to score candidate papers, and export graph-augmented relevance scores for the Hybrid Retrieval Agent.

**Architecture**:
- Nodes: Papers, Authors, Venues (typed)
- Edges: citation, co-authorship, same-venue, shared-keyword
- GNN: GraphSAGE or GAT over node embeddings → paper relevance score

**Outputs**:
- `data/citation_graph.gpickle` — NetworkX graph
- `data/gnn_checkpoint.pt` — trained GNN weights (Kaggle)
- `data/graph_scores.json` — paper_id → graph_relevance_score

In [ ]:
# ── Dependencies ────────────────────────────────────────────────────────────
import json
import time
import pickle
import requests
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

warnings.filterwarnings("ignore")

DATA_DIR = Path("../data")
EMB_DIR  = Path("../embeddings")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

import os
S2_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
S2_BASE    = "https://api.semanticscholar.org/graph/v1"
HEADERS    = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}

In [ ]:
# ── Load existing data ────────────────────────────────────────────────────────
papers_df = pd.read_csv(DATA_DIR / "papers_metadata.csv")
with open(EMB_DIR / "paper_id_mapping_e5.json") as f:
    id_mapping = json.load(f)
emb_matrix = np.load(EMB_DIR / "paper_embeddings_e5.npy")
print(f"Papers: {len(papers_df)}, Embeddings: {emb_matrix.shape}")

## 1. Fetch Citation Data from Semantic Scholar S2AG

In [ ]:
def fetch_citations(paper_id: str, limit: int = 50) -> List[Dict]:
    """Fetch papers that cite 'paper_id' from S2AG."""
    url = f"{S2_BASE}/paper/{paper_id}/citations"
    params = {"fields": "paperId,title,year", "limit": limit}
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=10)
        if resp.status_code == 200:
            return [item["citingPaper"] for item in resp.json().get("data", [])]
        elif resp.status_code == 429:
            time.sleep(2)
            return fetch_citations(paper_id, limit)
    except:
        pass
    return []


def fetch_references(paper_id: str, limit: int = 50) -> List[Dict]:
    """Fetch papers referenced by 'paper_id' from S2AG."""
    url = f"{S2_BASE}/paper/{paper_id}/references"
    params = {"fields": "paperId,title,year", "limit": limit}
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=10)
        if resp.status_code == 200:
            return [item["citedPaper"] for item in resp.json().get("data", [])]
        elif resp.status_code == 429:
            time.sleep(2)
            return fetch_references(paper_id, limit)
    except:
        pass
    return []

print("S2AG fetch helpers defined.")

## 2. Build Citation Graph (NetworkX)

In [ ]:
G = nx.DiGraph()

id_col = next((c for c in ["paper_id", "paperId", "id"] if c in papers_df.columns), None)

# ── Add paper nodes ──────────────────────────────────────────────────────────
for _, row in papers_df.iterrows():
    pid = str(row[id_col]) if id_col else str(row.name)
    attrs = {"type": "paper"}
    for col in ["title", "year", "venue"]:
        if col in row and pd.notna(row[col]):
            attrs[col] = str(row[col])
    if pid in id_mapping:
        attrs["emb_idx"] = id_mapping[pid]
    G.add_node(pid, **attrs)

print(f"Nodes added: {G.number_of_nodes()}")

In [ ]:
# ── Fetch and add citation edges (sample — use full corpus on Kaggle) ─────────
SAMPLE_SIZE = min(50, len(papers_df))  # set to len(papers_df) on Kaggle GPU
paper_ids = [str(row[id_col]) for _, row in papers_df.head(SAMPLE_SIZE).iterrows()] if id_col else []

edges_added = 0
if S2_API_KEY:
    for pid in tqdm(paper_ids[:SAMPLE_SIZE], desc="Fetching citation edges"):
        refs = fetch_references(pid, limit=30)
        for ref in refs:
            ref_id = ref.get("paperId")
            if ref_id:
                if not G.has_node(ref_id):
                    G.add_node(ref_id, type="paper", title=ref.get("title", ""), year=ref.get("year", ""))
                G.add_edge(pid, ref_id, type="cites")
                edges_added += 1
        time.sleep(0.3)
else:
    # Synthetic edges for development testing
    pids_list = list(G.nodes())[:SAMPLE_SIZE]
    np.random.seed(42)
    for i in range(min(200, len(pids_list))):
        src = pids_list[i]
        for tgt in np.random.choice(pids_list, size=3, replace=False):
            if src != tgt:
                G.add_edge(src, tgt, type="cites")
                edges_added += 1
    print("[Dev mode] Added synthetic citation edges.")

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# ── Add co-authorship edges (from local metadata) ─────────────────────────────
author_col = next((c for c in ["authors", "author"] if c in papers_df.columns), None)
if author_col:
    from itertools import combinations
    author_to_papers = defaultdict(list)
    for _, row in papers_df.iterrows():
        pid = str(row[id_col]) if id_col else str(row.name)
        authors_str = str(row.get(author_col, ""))
        for author in authors_str.split(",")[:5]:  # cap at 5 authors
            author_to_papers[author.strip()].append(pid)

    coauth_edges = 0
    for author, pids in author_to_papers.items():
        if len(pids) > 1:
            for p1, p2 in combinations(pids[:10], 2):
                if not G.has_edge(p1, p2):
                    G.add_edge(p1, p2, type="co_author")
                    coauth_edges += 1

    print(f"Co-authorship edges added: {coauth_edges}")
    print(f"Graph now: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# ── Compute graph centrality metrics ─────────────────────────────────────────
subgraph = G.subgraph(list(G.nodes())[:500])  # compute on sample

pagerank = nx.pagerank(subgraph, alpha=0.85)
in_degree = dict(subgraph.in_degree())

# Attach to graph nodes
nx.set_node_attributes(G, pagerank, "pagerank")
nx.set_node_attributes(G, in_degree, "in_degree")

pr_values = list(pagerank.values())
print(f"PageRank — mean: {np.mean(pr_values):.5f}, max: {np.max(pr_values):.5f}")

In [ ]:
# ── Save graph ────────────────────────────────────────────────────────────────
graph_path = DATA_DIR / "citation_graph.gpickle"
with open(graph_path, "wb") as f:
    pickle.dump(G, f)
print(f"Graph saved: {graph_path}")

## 3. GNN Scorer — GraphSAGE Implementation

In [ ]:
# ── Check PyTorch Geometric availability ─────────────────────────────────────
try:
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv, GATConv
    PYG_AVAILABLE = True
    print("PyTorch Geometric available.")
except ImportError:
    PYG_AVAILABLE = False
    print("PyTorch Geometric not installed. Using manual GNN implementation.")

In [ ]:
class GraphSAGEScorer(nn.Module):
    """Simple 2-layer GraphSAGE paper relevance scorer."""

    def __init__(self, in_dim: int, hidden_dim: int = 256, out_dim: int = 64):
        super().__init__()
        self.fc1  = nn.Linear(in_dim * 2, hidden_dim)  # concat self + mean-neighbor
        self.fc2  = nn.Linear(hidden_dim * 2, out_dim)
        self.score = nn.Linear(out_dim, 1)
        self.dropout = nn.Dropout(0.2)

    def aggregate_neighbors(self, node_embs: torch.Tensor,
                             adj_matrix: torch.Tensor) -> torch.Tensor:
        """Mean aggregation: sum over normalized adjacency."""
        # Normalize row-wise
        deg = adj_matrix.sum(dim=1, keepdim=True).clamp(min=1)
        agg = (adj_matrix / deg) @ node_embs
        return agg

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # Layer 1
        neigh1 = self.aggregate_neighbors(x, adj)
        h1 = F.relu(self.fc1(torch.cat([x, neigh1], dim=-1)))
        h1 = self.dropout(h1)
        h1 = F.normalize(h1, p=2, dim=-1)

        # Layer 2
        neigh2 = self.aggregate_neighbors(h1, adj)
        h2 = F.relu(self.fc2(torch.cat([h1, neigh2], dim=-1)))
        h2 = self.dropout(h2)
        h2 = F.normalize(h2, p=2, dim=-1)

        return self.score(h2).squeeze(-1)  # (N,)


print("GraphSAGEScorer defined.")

In [ ]:
# ── Build node feature matrix from embeddings ─────────────────────────────────
nodes_in_emb = [nid for nid in G.nodes() if str(nid) in id_mapping]
print(f"Nodes with embeddings: {len(nodes_in_emb)} / {G.number_of_nodes()}")

# Build feature matrix X: only for nodes with embeddings
node_idx = {nid: i for i, nid in enumerate(nodes_in_emb)}
X = np.array([emb_matrix[id_mapping[str(nid)]] for nid in nodes_in_emb], dtype=np.float32)
X_tensor = torch.tensor(X, device=DEVICE)

# Build adjacency matrix (subgraph of nodes with embeddings)
N = len(nodes_in_emb)
adj = np.zeros((N, N), dtype=np.float32)
for src, dst in G.edges():
    if str(src) in node_idx and str(dst) in node_idx:
        i, j = node_idx[str(src)], node_idx[str(dst)]
        adj[i, j] = 1.0
        adj[j, i] = 1.0  # undirected for aggregation

# Add self-loops
np.fill_diagonal(adj, 1.0)
adj_tensor = torch.tensor(adj, device=DEVICE)

print(f"Feature matrix: {X_tensor.shape}")
print(f"Adjacency matrix: {adj_tensor.shape}, density: {adj.mean():.4f}")

In [ ]:
# ── Initialize GNN and compute scores (inference mode, no training labels needed for scoring) ──
# NOTE: For training with preference pairs, see Notebook 08 (DPO pipeline)
# Here we use the GNN in zero-shot scoring mode: score = PageRank-weighted embedding norm

IN_DIM = X_tensor.shape[1]
gnn_model = GraphSAGEScorer(in_dim=IN_DIM, hidden_dim=256, out_dim=64).to(DEVICE)
print(f"GNN parameters: {sum(p.numel() for p in gnn_model.parameters()):,}")

# Zero-shot scores (random init → use as placeholder until trained on Kaggle)
gnn_model.eval()
with torch.no_grad():
    raw_scores = gnn_model(X_tensor, adj_tensor).cpu().numpy()

# Scale scores to [0, 1]
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
norm_scores = scaler.fit_transform(raw_scores.reshape(-1, 1)).flatten()

graph_scores = {str(nid): float(norm_scores[i]) for i, nid in enumerate(nodes_in_emb)}
print(f"Graph scores computed for {len(graph_scores)} papers")
print(f"Score stats — mean: {np.mean(norm_scores):.3f}, std: {np.std(norm_scores):.3f}")

In [ ]:
# ── Save GNN model and scores ─────────────────────────────────────────────────
torch.save(gnn_model.state_dict(), DATA_DIR / "gnn_checkpoint.pt")
with open(DATA_DIR / "graph_scores.json", "w") as f:
    json.dump(graph_scores, f)
print("Saved: gnn_checkpoint.pt, graph_scores.json")

## 4. Graph-Augmented Relevance Scoring Function

$$\text{Score}(p, u) = \lambda_1 \cdot \text{sem}(p, u) + \lambda_2 \cdot \text{gnn}(p) + \lambda_3 \cdot \text{pagerank}(p)$$

In [ ]:
def graph_augmented_score(paper_id: str, semantic_score: float,
                           lambda1: float = 0.6, lambda2: float = 0.3, lambda3: float = 0.1) -> float:
    """Combine semantic similarity with graph-based scores."""
    gnn_score  = graph_scores.get(str(paper_id), 0.0)
    pr_score   = pagerank.get(str(paper_id), 0.0) * 1000  # normalize PageRank
    pr_score   = min(pr_score, 1.0)
    return lambda1 * semantic_score + lambda2 * gnn_score + lambda3 * pr_score


# Example usage
sample_paper = list(graph_scores.keys())[0]
combined = graph_augmented_score(sample_paper, semantic_score=0.85)
print(f"Paper: {sample_paper}")
print(f"Combined graph-augmented score: {combined:.4f}")

## 5. Bridging Paper Discovery

Identify papers that bridge two otherwise disconnected clusters in the user's citation subgraph.

In [ ]:
def find_bridging_papers(user_paper_ids: List[str], candidate_ids: List[str],
                          top_k: int = 3) -> List[str]:
    """Find papers that maximally bridge two community clusters."""
    # Build user subgraph
    user_nodes = [pid for pid in user_paper_ids if G.has_node(pid)]
    if not user_nodes:
        return []
    subg = G.subgraph(user_nodes + candidate_ids)

    # Find connected components in undirected version
    undirected = subg.to_undirected()
    components = list(nx.connected_components(undirected))
    if len(components) < 2:
        return []

    # Candidates that appear in neighbors of multiple components are bridging
    bridging = []
    for cand in candidate_ids:
        if not G.has_node(cand):
            continue
        neighbor_comps = set()
        for neighbor in G.neighbors(cand):
            for ci, comp in enumerate(components):
                if neighbor in comp:
                    neighbor_comps.add(ci)
        if len(neighbor_comps) >= 2:
            bridging.append(cand)

    return bridging[:top_k]


# Test bridging discovery
sample_user_papers = list(G.nodes())[:5]
sample_candidates  = list(G.nodes())[5:20]
bridges = find_bridging_papers(sample_user_papers, sample_candidates)
print(f"Bridging papers found: {bridges}")

## 6. CiteCheck — Graph-Grounded Relational Verification (paper contribution §6)

Text verifiers (NLI, MiniCheck, RAGAS) check a claim against a *document* and cannot
confirm a **relationship between two papers** ("A builds on B", "A and B both address X").
**CiteCheck** verifies such relational claims against the **citation graph** `G` built above
(repurposing the graph from retrieval to verification).

`verify_relation(paper_a, paper_b, relation)` returns `supported / unsupported / unverifiable`
using citation direction, co-citation, shared references, and neighbor overlap.

> **Two things to calibrate before reporting results:**
> 1. **Node-id alignment** — benchmark `paper_id`s (arXiv ids for arXiv; S2 `paperId`s for
>    ACL/PubMed) must match `G`'s node ids. Add an id map if they differ (else claims fall to
>    `unverifiable`).
> 2. **Thresholds** (`CITECHECK_*`) — tune on the human-gold relational subset (from
>    `agentic-nb2`), exactly as MiniCheck thresholds are tuned.

In [ ]:
# ── CiteCheck verifier ────────────────────────────────────────────────────────
import networkx as nx

def _cc_year(n):
    try:
        return int(str(G.nodes[n].get("year", "")).strip()[:4])
    except Exception:
        return None

def _cc_cites(u, v):
    """True iff u cites v (directed 'cites' edge u->v)."""
    return G.has_edge(u, v) and G[u][v].get("type") == "cites"

def _cc_neighbors(n):
    return (set(G.predecessors(n)) | set(G.successors(n))) if n in G else set()

def _cc_co_citation(u, v):
    """Papers that cite BOTH u and v."""
    pu = {p for p in G.predecessors(u) if _cc_cites(p, u)} if u in G else set()
    pv = {p for p in G.predecessors(v) if _cc_cites(p, v)} if v in G else set()
    return pu & pv

def _cc_shared_refs(u, v):
    su = {s for s in G.successors(u) if _cc_cites(u, s)} if u in G else set()
    sv = {s for s in G.successors(v) if _cc_cites(v, s)} if v in G else set()
    return su & sv

# Thresholds — calibrate on the human-gold relational subset (agentic-nb2).
CITECHECK_COCITE_MIN  = 2      # min co-citations to support a relation
CITECHECK_JACCARD_MIN = 0.10   # min neighbor Jaccard for shares_topic

def verify_relation(paper_a, paper_b, relation):
    """Graph-grounded verification of a relational claim (A, relation, B)."""
    if not paper_a or not paper_b or paper_a not in G or paper_b not in G:
        return {"verdict": "unverifiable", "relation": relation,
                "reason": "node(s) missing from citation graph"}
    ya, yb = _cc_year(paper_a), _cc_year(paper_b)

    if relation in ("builds_on", "supersedes"):
        cites    = _cc_cites(paper_a, paper_b)
        order_ok = (ya is None or yb is None) or (ya >= yb)   # A no older than B
        ev = {"a_cites_b": cites, "year_a": ya, "year_b": yb}
        verdict = "supported" if (cites and order_ok) else "unsupported"

    elif relation == "compares":
        cc    = len(_cc_co_citation(paper_a, paper_b))
        cites = _cc_cites(paper_a, paper_b) or _cc_cites(paper_b, paper_a)
        ev = {"co_citation": cc, "either_cites": cites}
        verdict = "supported" if (cites or cc >= CITECHECK_COCITE_MIN) else "unsupported"

    else:  # shares_topic / co_addresses / default
        na, nb_ = _cc_neighbors(paper_a), _cc_neighbors(paper_b)
        jac = len(na & nb_) / len(na | nb_) if (na | nb_) else 0.0
        cc  = len(_cc_co_citation(paper_a, paper_b))
        sr  = len(_cc_shared_refs(paper_a, paper_b))
        ev = {"neighbor_jaccard": round(jac, 3), "co_citation": cc, "shared_refs": sr}
        verdict = ("supported" if (jac >= CITECHECK_JACCARD_MIN
                                   or cc >= CITECHECK_COCITE_MIN
                                   or sr >= CITECHECK_COCITE_MIN) else "unsupported")

    return {"verdict": verdict, "relation": relation, "evidence": ev}

# quick smoke test on two arbitrary connected nodes (if any)
_nodes = list(G.nodes())[:50]
_demo = next(((u, v) for u in _nodes for v in G.successors(u)), (None, None))
print("CiteCheck ready. demo verify_relation:",
      verify_relation(_demo[0], _demo[1], "builds_on") if _demo[0] else "(no edges in sample)")


In [ ]:
# ── Run CiteCheck over the benchmark's relational claims ─────────────────────
import sys, json
sys.path.append("..")
from config import SCIREQA_FULL
from collections import Counter

rel_claims = []
if SCIREQA_FULL.exists():
    with open(SCIREQA_FULL, encoding="utf-8") as f:
        for line in f:
            inst = json.loads(line)
            for lbl in inst.get("hallucination_labels", []):
                r = lbl.get("relational")
                if r and r.get("paper_a") and r.get("paper_b"):
                    rel_claims.append((inst["id"], lbl["claim_id"], r))
    print(f"Relational claims with two papers: {len(rel_claims)}")

    results = [{"id": iid, "claim_id": cid,
                **verify_relation(r["paper_a"], r["paper_b"], r["relation"])}
               for iid, cid, r in rel_claims]
    print("CiteCheck verdicts:", dict(Counter(x["verdict"] for x in results)))
    if results:
        cov = sum(1 for x in results if x["verdict"] != "unverifiable") / len(results)
        print(f"Graph coverage (adjudicable): {cov:.1%}  "
              f"(low coverage -> fix node-id alignment, see §7 note)")
    with open(DATA_DIR / "citecheck_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f)
    print(f"Saved {DATA_DIR / 'citecheck_results.json'}")
else:
    print(f"v1.1 benchmark not found at {SCIREQA_FULL} — run the benchmark constructor first.")


## 7. Summary

| Artifact | Description |
|---|---|
| `citation_graph.gpickle` | NetworkX DiGraph with citation + co-authorship edges |
| `gnn_checkpoint.pt` | GraphSAGE model weights (retrain on Kaggle with supervision) |
| `graph_scores.json` | paper_id → normalized GNN relevance score |

**Training note**: For supervised GNN training, use preference signals from `rated_papers.csv` as positive/negative paper pairs. Run full training on Kaggle P100/T4.  
**Next**: Notebook 05 — User Profile Agent